# MRI k-space: imaging with gradients and Fourier transforms

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/mri_kspace)

Magnetic-field gradients give every pixel its own precession phase, so the coil records the Fourier transform of the image: $s(\mathbf k) = \sum_x \rho(x)\,e^{-2\pi i\,\mathbf k\cdot\mathbf x}$ with $\mathbf k(t) = \tfrac{\gamma}{2\pi}\int \mathbf G\,dt$. Written as an encoding-matrix einsum it equals the FFT. Filling k-space line by line and inverting shows the image sharpen; skipping lines folds it over (aliasing), keeping only the centre blurs it with Gibbs ringing.

**Vector form**

$$s(\mathbf k) = \int \rho(\mathbf x)\,e^{-2\pi i\,\mathbf k\cdot\mathbf x}\,d^2x,\qquad \mathbf k(t) = \frac{\gamma}{2\pi}\int_0^t \mathbf G(t')\,dt',\qquad \rho = \mathcal F^{-1}[s]$$

**Index (tensor) form**

$$s_{uv} = E_{up}\,E_{vr}\,\rho_{pr},\qquad E_{up} = e^{-2\pi i\,up/N},\qquad \hat\rho_{pr} = \mathrm{IFFT}[\,m_{uv}\,s_{uv}\,]_{pr}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\rho_{pr}$ | `shepp_logan(N)` | (N, N) | proton-density image: pixel row $p$ (phase-encode direction), column $r$ (readout) |
| $k(t)$ | `readout_k(N, fov, G)` | (N,) | k-space position as the running integral of the gradient, $\tfrac{\gamma}{2\pi}\int G\,dt$; step $\Delta k = 1/\mathrm{FOV}$ |
| $E_{up}$ | `encoding_matrix(N)` | (N, N) complex | phase $e^{-2\pi i\,up/N}$ of pixel $p$ at k-space line $u$; the 2D matrix $E_{kx}$ factorises as $E_{up}E_{vr}$ |
| $s_{uv}$ | `einsum('up,vr,pr->uv', E, E, rho)` | (N, N) complex | repeated $p, r$ ⇒ summed: the measured k-space signal, identical to fft2(rho) |
| $m_{uv}$ | `line_mask(lines, N)` | (N, N) | sampling mask: 1 on acquired phase-encode lines |
| $\hat\rho_{pr}$ | `torch.fft.ifft2(mask * s)` | (N, N) | zero-filled reconstruction |
| $(A^H A + \lambda W_r)\,\rho_{:r} = A^H h_{:r}$ | `torch.linalg.solve(lhs, rhs)` | (N, N, N) | Tikhonov least squares per image column $r$, penalising signal outside the head |

Every other line ($R = 2$) halves the field of view: the zero-filled image is exactly $\tfrac12[\rho(p) + \rho(p + N/2)]$, a copy shifted by $\mathrm{FOV}/2$ that wraps around. Keeping only the central lines keeps the DC term, so total intensity is preserved, but loses the edges: the image blurs and rings (Gibbs). The regularised reconstruction solves a small least-squares problem for each image column with a penalty on signal outside a head mask estimated from the central lines; it reduces the error of the random scheme a little but cannot undo regular aliasing when the folded copies both land inside the head. Real scanners add multiple coils (parallel imaging) or sparsity priors (compressed sensing). The phantom is the classic Shepp–Logan head; illustrative only, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/mri_kspace/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import encode, encode_fft


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = params["size"]
    rho = tensors.shepp_logan(n, dev)  # (N, N) proton density
    E = tensors.encoding_matrix(n, dev)  # (N, N) complex
    k, dwell = tensors.readout_k(n, fov=0.24, gradient=10e-3)
    print(f"readout: G = 10 mT/m, FOV = 24 cm -> dwell time {dwell * 1e6:.2f} us, k from {k[0]:.1f} to {k[-1]:.1f} 1/m")

    s = encode(rho, E)  # (N, N) the measured signal, written as the encoding-matrix einsum
    print(f"max |einsum encoding - FFT| = {(s - encode_fft(rho)).abs().max():.2e}")

    masks, images = solve.filling(s, params["n_frames"])
    print(f"full sampling: max |reconstruction - phantom| = {(images[-1] - rho).abs().max():.2e}")
    results = solve.schemes(s, E, params["R"], params["lowpass"], params["random_frac"], params["lam"])
    for name, mask, zf, reg in results:
        print(
            f"{name:16s} lines {int(mask[:, 0].sum()):3d}/{n}  RMSE zero-filled {solve.rmse(zf, rho):.3f}"
            f"  regularised {solve.rmse(reg, rho):.3f}"
        )
    return plot.animate(s, rho, masks, images, results, out_dir / "mri_kspace.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/mri_kspace/equation.py
r"""Physics → tensor translation.

With gradients switched on, a spin at position x precesses at a frequency that depends on x, so by time t it has
picked up the phase 2\pi\,k(t)\cdot x with k(t) = \frac{\gamma}{2\pi}\int_0^t G\,dt'. The coil adds up all spins:

    s(k) = \sum_x \rho(x)\, e^{-2\pi i\, k\cdot x}

On an N × N grid (pixels p, r; k-space lines u, samples v) this is a linear map, the encoding matrix:

    s_{uv} = E_{up} E_{vr} \rho_{pr},      E_{up} = e^{-2\pi i u p / N}          (p, r summed)

which is exactly the 2D discrete Fourier transform, so s = FFT(ρ) and the image comes back as ρ = IFFT(s).
Missing lines (sampling mask m_{uv}) give the zero-filled image IFFT(m ⊙ s). With only some lines u ∈ U,
a regularised (Tikhonov) reconstruction solves, for every image column r, the least-squares problem

    (A^H A + \lambda W_r)\, \rho_{:r} = A^H h_{Ur} / \sqrt N,    A = E_{Up}/\sqrt N,   W_r = diag(1 outside the head)
"""

import torch


def encode(rho: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    r"""s_{uv} = E_{up} E_{vr} \rho_{pr}  (p, r summed).   rho: (N, N), E: (N, N) → (N, N) complex"""
    return torch.einsum("up,vr,pr->uv", E, E, rho.to(E.dtype))


def encode_fft(rho: torch.Tensor) -> torch.Tensor:
    """The same signal with the fast Fourier transform: O(N² log N) instead of O(N³)."""
    return torch.fft.fft2(rho)


def zero_filled(s: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    r"""\hat\rho = IFFT(m_{uv} s_{uv}): unmeasured samples set to zero.   s: (N, N), mask: (..., N, N) → (..., N, N)"""
    return torch.fft.ifft2(mask * s)


def regularised(
    s: torch.Tensor, lines: torch.Tensor, E: torch.Tensor, outside: torch.Tensor, lam: float
) -> torch.Tensor:
    r"""Solve (A^H A + \lambda W_r) \rho_{:r} = A^H h_{Ur} / \sqrt N for every column r (batched).   → (N, N) complex

    h_{ur} = IFFT_v(s_{uv}) = E_{up} \rho_{pr} is the data after transforming the fully sampled readout direction.
    W_r = diag(outside_{:r}) penalises signal where we know there is no tissue (outside the head); a tiny
    multiple of the identity keeps every system solvable.
    """
    n = E.shape[0]
    h = torch.fft.ifft(s[lines], dim=1)  # (M, N): row u = acquired line, column r = image column
    A = E[lines] / n**0.5  # (M, N) encoding along the phase-encode direction p
    AhA = A.conj().T @ A  # (N, N), shared by all columns
    W = torch.diag_embed(outside.T.to(E.dtype))  # (N_r, N_p, N_p): one diagonal weight matrix per column r
    eye = torch.eye(n, dtype=E.dtype, device=E.device)
    lhs = AhA + lam * W + 1e-6 * eye  # (N_r, N, N)
    rhs = (A.conj().T @ h / n**0.5).T[..., None]  # (N_r, N, 1): A^H h for each column
    return torch.linalg.solve(lhs, rhs)[..., 0].T  # (N_p, N_r)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/mri_kspace/tensors.py
"""Build the concrete tensors: Shepp–Logan phantom, encoding matrix, gradient-made k-space positions, sampling masks."""

import math

import torch

# Modified Shepp–Logan head phantom: (intensity, semi-axis a, semi-axis b, centre x, centre y, angle in degrees).
_ELLIPSES = [
    (1.0, 0.69, 0.92, 0.0, 0.0, 0),
    (-0.8, 0.6624, 0.874, 0.0, -0.0184, 0),
    (-0.2, 0.11, 0.31, 0.22, 0.0, -18),
    (-0.2, 0.16, 0.41, -0.22, 0.0, 18),
    (0.1, 0.21, 0.25, 0.0, 0.35, 0),
    (0.1, 0.046, 0.046, 0.0, 0.1, 0),
    (0.1, 0.046, 0.046, 0.0, -0.1, 0),
    (0.1, 0.046, 0.023, -0.08, -0.605, 0),
    (0.1, 0.023, 0.023, 0.0, -0.606, 0),
    (0.1, 0.023, 0.046, 0.06, -0.605, 0),
]


def shepp_logan(n: int, device: torch.device) -> torch.Tensor:
    """Proton-density image ρ_{pr} (row p, column r) of the Shepp–Logan phantom. Shape (N, N) float64."""
    c = torch.linspace(-1, 1, n, dtype=torch.float64, device=device)
    y, x = torch.meshgrid(-c, c, indexing="ij")  # row 0 at the top (y = +1)
    rho = torch.zeros(n, n, dtype=torch.float64, device=device)
    for value, a, b, x0, y0, deg in _ELLIPSES:
        th = math.radians(deg)
        xr = (x - x0) * math.cos(th) + (y - y0) * math.sin(th)
        yr = -(x - x0) * math.sin(th) + (y - y0) * math.cos(th)
        rho += value * ((xr / a) ** 2 + (yr / b) ** 2 <= 1).double()
    return rho


def encoding_matrix(n: int, device: torch.device) -> torch.Tensor:
    """E_{up} = exp(−2πi u p / N): the phase a spin at pixel p picks up at k-space line u. Shape (N, N) complex.

    In 2D the encoding is separable, E_{(u,v),(p,r)} = E_{up} E_{vr}, so the full (N², N²) matrix is never built.
    """
    idx = torch.arange(n, dtype=torch.float64, device=device)
    return torch.exp(-2j * math.pi * torch.outer(idx, idx) / n)


def readout_k(n: int, fov: float, gradient: float, gamma_bar: float = 42.58e6) -> tuple[torch.Tensor, float]:
    """k(t) = γ/(2π) ∫ G dt along one readout, sampled every dwell time Δt.  Returns (k (N,) in 1/m, Δt in s).

    The k-space step must be Δk = 1/FOV to cover the field of view, so Δt = Δk / (γ̄ G).
    A pre-phasing gradient lobe first moves k to −N/2·Δk, then the readout gradient G sweeps it at constant speed.
    """
    dk = 1.0 / fov
    dwell = dk / (gamma_bar * gradient)
    G = torch.full((n,), gradient, dtype=torch.float64)  # (N,) constant readout gradient
    area = torch.cumsum(G * dwell, dim=0) - G * dwell  # (N,) ∫_0^{t_n} G dt at the start of each sample
    k = -n / 2 * dk + gamma_bar * area  # (N,) runs from −N/2·Δk to (N/2 − 1)·Δk
    return k, dwell


def centric_order(n: int) -> torch.Tensor:
    """Phase-encode lines u in the order 0, 1, −1, 2, −2, … (centre of k-space first), as FFT indices. (N,)"""
    signed = torch.arange(n) - n // 2  # −N/2 … N/2 − 1
    order = signed[torch.argsort(signed.abs() * 2 - (signed > 0).long(), stable=True)]
    return order % n


def line_mask(lines: torch.Tensor, n: int, device: torch.device) -> torch.Tensor:
    """Sampling mask m_{uv}: 1 on every acquired line u (all readout samples v), 0 elsewhere. (N, N)"""
    mask = torch.zeros(n, n, dtype=torch.float64, device=device)
    mask[lines] = 1.0
    return mask


def scheme_lines(scheme: str, n: int, value: float, seed: int = 0) -> torch.Tensor:
    """Acquired line indices (FFT order) for an undersampling scheme.

    'skip': every R-th line (R = value), 'lowpass': only the central fraction value of lines,
    'random': a fully sampled centre plus random outer lines, value = fraction kept (fixed seed).
    """
    signed = torch.arange(n) - n // 2
    if scheme == "skip":
        lines = torch.arange(0, n, int(value))
    elif scheme == "lowpass":
        keep = max(1, round(value * n))
        lines = signed[torch.argsort(signed.abs(), stable=True)[:keep]] % n
    else:
        centre = signed.abs() <= n // 16
        outer = torch.nonzero(~centre).flatten()
        g = torch.Generator().manual_seed(seed)
        extra = max(0, round(value * n) - int(centre.sum()))
        chosen = outer[torch.randperm(len(outer), generator=g)[:extra]]
        lines = torch.cat([signed[centre] % n, signed[chosen] % n])
    return torch.sort(lines).values


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/mri_kspace/solve.py
"""Fill k-space line by line, reconstruct after every line, and compare undersampling schemes."""

import torch

from . import tensors
from .equation import regularised, zero_filled


def filling(s: torch.Tensor, n_frames: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Centre-out Cartesian acquisition: masks (F, N, N) after each frame and the zero-filled images (F, N, N)."""
    n = s.shape[0]
    order = tensors.centric_order(n)  # (N,) lines in acquisition order
    counts = torch.linspace(1, n, n_frames).round().long()  # lines acquired by each frame
    masks = torch.stack([tensors.line_mask(order[:c], n, s.device) for c in counts])  # (F, N, N)
    return masks, zero_filled(s, masks).abs()  # one batched inverse FFT for all frames


def support_weights(s: torch.Tensor) -> torch.Tensor:
    """outside_{pr}: 1 where a blurry image from the central 1/8 of k-space shows no tissue, else 0. (N, N)"""
    n = s.shape[0]
    low = zero_filled(s, tensors.line_mask(tensors.scheme_lines("lowpass", n, 1 / 8), n, s.device)).abs()
    inside = (low > 0.15 * low.max()).double()
    inside = torch.nn.functional.max_pool2d(inside[None, None], 5, 1, 2)[0, 0]  # grow the head mask by 2 px
    return 1.0 - inside


def schemes(
    s: torch.Tensor, E: torch.Tensor, R: int, lowpass: float, random_frac: float, lam: float
) -> list[tuple[str, torch.Tensor, torch.Tensor, torch.Tensor]]:
    """(name, mask, zero-filled image, regularised image) for skip-R, low-pass and random undersampling."""
    n = s.shape[0]
    outside = support_weights(s)
    ordinal = {2: "2nd", 3: "3rd"}.get(R, f"{R}th")
    out = []
    for name, scheme, value in [
        (f"every {ordinal} line", "skip", R),
        (f"central {lowpass:.0%} only", "lowpass", lowpass),
        (f"random {random_frac:.0%}", "random", random_frac),
    ]:
        lines = tensors.scheme_lines(scheme, n, value)
        mask = tensors.line_mask(lines, n, s.device)
        out.append((name, mask, zero_filled(s, mask).abs(), regularised(s, lines, E, outside, lam).abs()))
    return out


def rmse(image: torch.Tensor, truth: torch.Tensor) -> float:
    return float((image - truth).pow(2).mean().sqrt())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/mri_kspace/plot.py
"""Animate k-space filling with the sharpening reconstruction, then compare three undersampling schemes."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def _kspace_view(s: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """log10 |s| with the zero frequency in the centre; unmeasured samples masked out (drawn grey)."""
    view = np.fft.fftshift(np.log10(np.abs(s) + 1e-2))
    return np.ma.masked_where(np.fft.fftshift(mask) == 0, view)


def animate(
    s: torch.Tensor,
    rho: torch.Tensor,
    masks: torch.Tensor,
    images: torch.Tensor,
    results: list[tuple[str, torch.Tensor, torch.Tensor, torch.Tensor]],
    path: Path,
) -> Path:
    s_np, truth = s.cpu().numpy(), rho.cpu().numpy()
    masks_np, images_np = masks.cpu().numpy(), images.cpu().numpy()
    n_fill = len(masks_np)
    hold = max(1, n_fill // 4)  # frames spent on each undersampling scheme
    total = n_fill + hold * len(results)

    fig, axes = plt.subplots(1, 3, figsize=(7.8, 3.2))
    cmap_k = plt.get_cmap("viridis").with_extremes(bad="#e5e7eb")  # unmeasured samples in light grey
    full = np.fft.fftshift(np.log10(np.abs(s_np) + 1e-2))
    k_im = axes[0].imshow(_kspace_view(s_np, masks_np[0]), cmap=cmap_k, vmin=full.min(), vmax=full.max())
    rec_im = axes[1].imshow(images_np[0], cmap="gray", vmin=0, vmax=0.6)
    ref_im = axes[2].imshow(truth, cmap="gray", vmin=0, vmax=0.6)
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    titles = [ax.set_title("", fontsize=9) for ax in axes]
    banner = fig.suptitle("", fontsize=10)

    def update(step: int):
        if step < n_fill:
            lines = int(masks_np[step][:, 0].sum())
            k_im.set_data(_kspace_view(s_np, masks_np[step]))
            rec_im.set_data(images_np[step])
            ref_im.set_data(truth)
            banner.set_text(f"Cartesian filling, centre out: {lines}/{len(truth)} phase-encode lines")
            titles[0].set_text("k-space  log|s(k)|")
            titles[1].set_text("inverse FFT of lines so far")
            titles[2].set_text("phantom ρ(x)")
            return
        name, mask, zf, reg = results[min((step - n_fill) // hold, len(results) - 1)]
        k_im.set_data(_kspace_view(s_np, mask.cpu().numpy()))
        rec_im.set_data(zf.cpu().numpy())
        ref_im.set_data(reg.cpu().numpy())
        banner.set_text(f"Undersampled: {name}")
        titles[0].set_text("acquired k-space")
        titles[1].set_text("zero-filled IFFT")
        titles[2].set_text("regularised least squares")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(total, total), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'size': 64,
    'R': 2,
    'lowpass': 0.25,
    'random_frac': 0.4,
    'lam': 10.0,
    'n_frames': 48
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.mri_kspace")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.mri_kspace.main")
problem_dir = Path("problems/medical/mri_kspace")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
